In [1]:
# gold_etf_feature_fetcher.py

import warnings
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import yfinance as yf


warnings.filterwarnings("ignore")


# ============================================================
# 1. 取得対象ティッカー
# ============================================================

TICKERS: Dict[str, str] = {
    # --------------------------------------------------------
    # Gold / Precious metals
    # --------------------------------------------------------
    "gold_futures": "GC=F",
    "gold_etf_gld": "GLD",
    "gold_etf_iau": "IAU",
    "gold_etf_jp_1540": "1540.T",
    "gold_etf_jp_1326": "1326.T",
    "gold_miners_gdx": "GDX",
    "junior_gold_miners_gdxj": "GDXJ",
    "silver_etf_slv": "SLV",
    "silver_miners_sil": "SIL",
    # --------------------------------------------------------
    # FX / Dollar
    # --------------------------------------------------------
    "usd_jpy": "JPY=X",
    "dxy": "DX-Y.NYB",
    # --------------------------------------------------------
    # US equity indices / ETFs
    # --------------------------------------------------------
    "sp500": "^GSPC",
    "spy": "SPY",
    "nasdaq100": "^NDX",
    "qqq": "QQQ",
    "dow": "^DJI",
    "dia": "DIA",
    "russell2000": "^RUT",
    "iwm": "IWM",
    # --------------------------------------------------------
    # Volatility
    # --------------------------------------------------------
    "vix": "^VIX",
    # --------------------------------------------------------
    # US sector ETFs
    # --------------------------------------------------------
    "materials_xlb": "XLB",
    "energy_xle": "XLE",
    "financials_xlf": "XLF",
    "technology_xlk": "XLK",
    "utilities_xlu": "XLU",
    "consumer_staples_xlp": "XLP",
    "consumer_discretionary_xly": "XLY",
    "industrials_xli": "XLI",
    "healthcare_xlv": "XLV",
    "communication_xlc": "XLC",
    "real_estate_xlre": "XLRE",
    # --------------------------------------------------------
    # Commodity-related ETFs / futures
    # --------------------------------------------------------
    "metals_mining_xme": "XME",
    "copper_miners_copx": "COPX",
    "wti_crude": "CL=F",
    "brent_crude": "BZ=F",
    "copper_futures": "HG=F",
    # --------------------------------------------------------
    # Resource stocks
    # --------------------------------------------------------
    "freeport_fcx": "FCX",
    "bhp": "BHP",
    "rio_tinto": "RIO",
    "vale": "VALE",
    # --------------------------------------------------------
    # Gold mining individual stocks
    # --------------------------------------------------------
    "newmont_nem": "NEM",
    "barrick_gold": "GOLD",
    "agnico_eagle": "AEM",
    "kinross": "KGC",
    "gold_fields": "GFI",
    "anglogold": "AU",
    "wheaton": "WPM",
    "franco_nevada": "FNV",
    "royal_gold": "RGLD",
    # --------------------------------------------------------
    # Financials / credit-sensitive stocks
    # --------------------------------------------------------
    "jpmorgan": "JPM",
    "bank_of_america": "BAC",
    "goldman_sachs": "GS",
    "morgan_stanley": "MS",
    "blackrock": "BLK",
    # --------------------------------------------------------
    # Large growth / risk-on proxies
    # --------------------------------------------------------
    "nvidia": "NVDA",
    "microsoft": "MSFT",
    "apple": "AAPL",
    "amazon": "AMZN",
    "alphabet": "GOOGL",
    "meta": "META",
    "tesla": "TSLA",
    # --------------------------------------------------------
    # Japan market
    # --------------------------------------------------------
    "nikkei225": "^N225",
    "topix": "^TOPX",
    # 国内業種指数はYahooで安定しないことが多いので、ETFで代替
    "japan_bank_etf": "1615.T",
    "japan_reit_etf": "1476.T",
}


# ============================================================
# 2. データ取得
# ============================================================


def download_ohlcv(
    tickers: Dict[str, str],
    start: str = "2015-01-01",
    end: str | None = None,
    interval: str = "1d",
) -> pd.DataFrame:
    """
    yfinanceからOHLCVを取得する。
    返り値は columns = [field, name] のMultiIndex。
    """

    symbols: List[str] = list(tickers.values())
    name_by_symbol = {v: k for k, v in tickers.items()}

    df = yf.download(
        symbols,
        start=start,
        end=end,
        interval=interval,
        auto_adjust=True,
        group_by="column",
        threads=True,
        progress=True,
    )

    if df.empty:
        raise ValueError(
            "データ取得に失敗しました。ティッカー、期間、ネットワークを確認してください。"
        )

    # yfinanceの列は通常 MultiIndex: Price field x Ticker
    if isinstance(df.columns, pd.MultiIndex):
        # ticker symbolをわかりやすい名前に変換
        new_cols = []
        for field, symbol in df.columns:
            new_cols.append((field, name_by_symbol.get(symbol, symbol)))
        df.columns = pd.MultiIndex.from_tuples(new_cols, names=["field", "asset"])
    else:
        # 単一ティッカー時の保険
        only_name = list(tickers.keys())[0]
        df.columns = pd.MultiIndex.from_product(
            [df.columns, [only_name]], names=["field", "asset"]
        )

    return df.sort_index()


def extract_close(ohlcv: pd.DataFrame) -> pd.DataFrame:
    """
    OHLCVからClose価格だけ抽出。
    """
    if "Close" not in ohlcv.columns.get_level_values("field"):
        raise ValueError("Close列が見つかりません。")

    close = ohlcv["Close"].copy()
    close = close.dropna(axis=1, how="all")
    close.index = pd.to_datetime(close.index)
    return close.sort_index()


def extract_volume(ohlcv: pd.DataFrame) -> pd.DataFrame:
    """
    OHLCVからVolumeだけ抽出。
    為替や指数などVolumeがないものは欠損になる。
    """
    if "Volume" not in ohlcv.columns.get_level_values("field"):
        return pd.DataFrame(index=ohlcv.index)

    volume = ohlcv["Volume"].copy()
    volume = volume.dropna(axis=1, how="all")
    volume.index = pd.to_datetime(volume.index)
    return volume.sort_index()


# ============================================================
# 3. 特徴量作成
# ============================================================


def make_return_features(
    close: pd.DataFrame, windows=(1, 3, 5, 10, 20, 60)
) -> pd.DataFrame:
    features = []

    for w in windows:
        ret = close.pct_change(w)
        ret.columns = [f"{c}_ret_{w}d" for c in ret.columns]
        features.append(ret)

    return pd.concat(features, axis=1)


def make_volatility_features(close: pd.DataFrame, windows=(5, 20, 60)) -> pd.DataFrame:
    daily_ret = close.pct_change()
    features = []

    for w in windows:
        vol = daily_ret.rolling(w).std() * np.sqrt(252)
        vol.columns = [f"{c}_vol_{w}d" for c in vol.columns]
        features.append(vol)

    return pd.concat(features, axis=1)


def make_ma_gap_features(close: pd.DataFrame, windows=(5, 20, 60, 120)) -> pd.DataFrame:
    features = []

    for w in windows:
        ma = close.rolling(w).mean()
        gap = close / ma - 1
        gap.columns = [f"{c}_ma_gap_{w}d" for c in gap.columns]
        features.append(gap)

    return pd.concat(features, axis=1)


def make_drawdown_features(close: pd.DataFrame, windows=(20, 60, 120)) -> pd.DataFrame:
    features = []

    for w in windows:
        rolling_high = close.rolling(w).max()
        rolling_low = close.rolling(w).min()

        dd_from_high = close / rolling_high - 1
        up_from_low = close / rolling_low - 1

        dd_from_high.columns = [f"{c}_dd_from_{w}d_high" for c in dd_from_high.columns]
        up_from_low.columns = [f"{c}_up_from_{w}d_low" for c in up_from_low.columns]

        features.extend([dd_from_high, up_from_low])

    return pd.concat(features, axis=1)


def make_volume_features(volume: pd.DataFrame, windows=(5, 20)) -> pd.DataFrame:
    if volume.empty:
        return pd.DataFrame(index=volume.index)

    features = []

    volume_chg_1d = volume.pct_change()
    volume_chg_1d.columns = [f"{c}_volume_chg_1d" for c in volume_chg_1d.columns]
    features.append(volume_chg_1d)

    for w in windows:
        avg_volume = volume.rolling(w).mean()
        volume_ratio = volume / avg_volume - 1
        volume_ratio.columns = [f"{c}_volume_ratio_{w}d" for c in volume_ratio.columns]
        features.append(volume_ratio)

    return pd.concat(features, axis=1)


def safe_ratio(
    close: pd.DataFrame, numerator: str, denominator: str, name: str
) -> pd.Series:
    if numerator not in close.columns or denominator not in close.columns:
        return pd.Series(index=close.index, name=name, dtype=float)
    return (close[numerator] / close[denominator]).rename(name)


def make_relative_features(close: pd.DataFrame) -> pd.DataFrame:
    """
    金ETFモデルで効きそうな相対価格・相対パフォーマンス。
    """
    ratios = pd.DataFrame(index=close.index)

    ratio_pairs = [
        ("gold_miners_gdx", "gold_etf_gld", "gdx_div_gld"),
        ("junior_gold_miners_gdxj", "gold_miners_gdx", "gdxj_div_gdx"),
        ("silver_etf_slv", "gold_etf_gld", "slv_div_gld"),
        ("materials_xlb", "spy", "xlb_div_spy"),
        ("energy_xle", "spy", "xle_div_spy"),
        ("financials_xlf", "spy", "xlf_div_spy"),
        ("technology_xlk", "spy", "xlk_div_spy"),
        ("utilities_xlu", "spy", "xlu_div_spy"),
        ("consumer_staples_xlp", "spy", "xlp_div_spy"),
        ("qqq", "spy", "qqq_div_spy"),
        ("japan_reit_etf", "topix", "jreit_div_topix"),
        ("japan_bank_etf", "topix", "japan_bank_div_topix"),
    ]

    for numerator, denominator, name in ratio_pairs:
        ratios[name] = safe_ratio(close, numerator, denominator, name)

    # 相対価格そのものより、変化率の方がモデルに入れやすい
    features = []

    for w in [1, 3, 5, 10, 20]:
        ret = ratios.pct_change(w)
        ret.columns = [f"{c}_ret_{w}d" for c in ret.columns]
        features.append(ret)

    for w in [20, 60]:
        ma = ratios.rolling(w).mean()
        gap = ratios / ma - 1
        gap.columns = [f"{c}_ma_gap_{w}d" for c in gap.columns]
        features.append(gap)

    return pd.concat(features, axis=1)


def make_calendar_features(index: pd.DatetimeIndex) -> pd.DataFrame:
    """
    月内の購入タイミング判断に重要なカレンダー特徴量。
    """
    cal = pd.DataFrame(index=index)
    cal["year"] = index.year
    cal["month"] = index.month
    cal["day"] = index.day
    cal["dayofweek"] = index.dayofweek

    # 月内営業日番号
    month_key = pd.Series(index.to_period("M"), index=index)
    cal["business_day_in_month"] = month_key.groupby(month_key).cumcount() + 1

    # 月内営業日数
    business_days_in_month = cal.groupby(["year", "month"])[
        "business_day_in_month"
    ].transform("max")
    cal["business_days_in_month"] = business_days_in_month

    # 月内残営業日数
    cal["remaining_business_days_in_month"] = (
        cal["business_days_in_month"] - cal["business_day_in_month"]
    )

    cal["month_progress"] = cal["business_day_in_month"] / cal["business_days_in_month"]
    cal["is_month_start_area"] = (cal["business_day_in_month"] <= 3).astype(int)
    cal["is_month_end_area"] = (cal["remaining_business_days_in_month"] <= 3).astype(
        int
    )

    return cal


def build_features(close: pd.DataFrame, volume: pd.DataFrame) -> pd.DataFrame:
    feature_blocks = [
        make_return_features(close),
        make_volatility_features(close),
        make_ma_gap_features(close),
        make_drawdown_features(close),
        make_volume_features(volume),
        make_relative_features(close),
        make_calendar_features(close.index),
    ]

    features = pd.concat(feature_blocks, axis=1)
    features = features.replace([np.inf, -np.inf], np.nan)

    return features


# ============================================================
# 4. 目的変数作成
# ============================================================


def make_target_future_lower_probability(
    close: pd.DataFrame,
    target_asset: str = "gold_etf_jp_1540",
) -> pd.DataFrame:
    """
    目的変数:
    「月末までに、今日より安い日が来たか」

    y_future_lower = 1:
        今月中の将来日で今日より安い価格があった

    y_future_lower = 0:
        今日より安い日は今月中になかった

    Buy Probability = 1 - Pr(y_future_lower = 1)
    """

    if target_asset not in close.columns:
        raise ValueError(
            f"{target_asset} がcloseに存在しません。取得対象を確認してください。"
        )

    price = close[target_asset].copy()
    df = pd.DataFrame(index=price.index)
    df["price"] = price
    df["year_month"] = price.index.to_period("M")

    future_min_list = []

    for _, g in df.groupby("year_month"):
        # 翌日以降から月末までの最小値
        # reverse rolling min のように処理
        future_min = g["price"][::-1].cummin()[::-1].shift(-1)
        future_min_list.append(future_min)

    df["future_min_until_month_end"] = pd.concat(future_min_list).sort_index()
    df["future_max_drawdown_until_month_end"] = (
        df["future_min_until_month_end"] / df["price"] - 1
    )

    df["y_future_lower"] = (df["future_min_until_month_end"] < df["price"]).astype(
        float
    )

    # 月末最終営業日は未来がないので目的変数なし
    df.loc[df["future_min_until_month_end"].isna(), "y_future_lower"] = np.nan

    return df[
        [
            "price",
            "future_min_until_month_end",
            "future_max_drawdown_until_month_end",
            "y_future_lower",
        ]
    ]


# ============================================================
# 5. メイン処理
# ============================================================


def main():
    output_dir = Path("gold_etf_data")
    output_dir.mkdir(exist_ok=True)

    start = "2015-01-01"
    end = None

    print("Downloading market data...")
    ohlcv = download_ohlcv(TICKERS, start=start, end=end)

    print("Extracting close and volume...")
    close = extract_close(ohlcv)
    volume = extract_volume(ohlcv)

    print("Building features...")
    features = build_features(close, volume)

    print("Building target...")
    target = make_target_future_lower_probability(
        close,
        target_asset="gold_etf_jp_1540",
    )

    dataset = features.join(target, how="left")

    # 学習に使うなら目的変数がある行だけに絞る
    train_dataset = dataset.dropna(subset=["y_future_lower"]).copy()

    # 保存
    ohlcv.to_csv(output_dir / "raw_ohlcv.csv", encoding="utf-8-sig")
    close.to_csv(output_dir / "close_prices.csv", encoding="utf-8-sig")
    volume.to_csv(output_dir / "volumes.csv", encoding="utf-8-sig")
    features.to_csv(output_dir / "features.csv", encoding="utf-8-sig")
    target.to_csv(output_dir / "target.csv", encoding="utf-8-sig")
    dataset.to_csv(output_dir / "dataset_all.csv", encoding="utf-8-sig")
    train_dataset.to_csv(output_dir / "dataset_train.csv", encoding="utf-8-sig")

    print("Done.")
    print(f"Close shape: {close.shape}")
    print(f"Features shape: {features.shape}")
    print(f"Train dataset shape: {train_dataset.shape}")
    print(f"Saved to: {output_dir.resolve()}")

    # 取得できなかった系列の確認
    missing_assets = [name for name in TICKERS.keys() if name not in close.columns]

    if missing_assets:
        print("\nWarning: Some assets were not downloaded:")
        for asset in missing_assets:
            print(f"  - {asset}: {TICKERS[asset]}")


if __name__ == "__main__":
    main()

$^TOPX: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-05-18)
[*********************100%***********************]  65 of 65 completed

1 Failed download:
['^TOPX']: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-05-18)


Extracting close and volume...
Building features...
Building target...
Done.
Close shape: (2966, 64)
Features shape: (2966, 1502)
Train dataset shape: (2673, 1506)
Saved to: C:\Users\fuben\github\gold-buy-timing-optimizer\gold_etf_data

  - topix: ^TOPX


In [3]:
close = pd.read_csv("gold_etf_data/close_prices.csv")
close

,Date,gold_etf_jp_1326,japan_reit_etf,gold_etf_jp_1540,japan_bank_etf,apple,agnico_eagle,amazon,anglogold,bank_of_america,...,utilities_xlu,healthcare_xlv,consumer_discretionary_xly,metals_mining_xme,dow,sp500,nikkei225,nasdaq100,russell2000,vix
0,2015-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-01-02,NaN,NaN,NaN,NaN,24.192608,22.215916,15.426000,7.346209,14.123799,...,16.511276,57.057846,31.636652,26.510984,17832.990234,2058.199951,NaN,4230.240234,1198.800049,17.790001
2,2015-01-05,13820.0,NaN,4540.0,131.179596,23.511057,22.586184,15.109500,7.672522,13.713499,...,16.309406,56.766861,31.031580,25.531956,17501.650391,2020.579956,17408.710938,4160.959961,1181.349976,19.920000
3,2015-01-06,13790.0,NaN,4510.0,127.763458,23.513269,23.886227,14.764500,8.149440,13.303198,...,16.319849,56.575638,30.722391,25.179852,17371.640625,2002.609985,16883.189453,4110.830078,1161.310059,21.120001
4,2015-01-07,13850.0,NaN,4540.0,126.397018,23.842979,23.400764,14.921000,7.998835,13.366322,...,16.479950,57.905853,31.208220,25.291492,17584.519531,2025.900024,16885.330078,4160.000000,1175.969971,19.309999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2961,2026-05-12,67860.0,1899.0,22150.0,640.799988,294.799988,197.050003,265.820007,104.330002,50.779999,...,45.189999,145.850006,118.290001,123.599998,49760.558594,7400.959961,62742.570312,29064.800781,2842.830078,17.990000
2962,2026-05-13,68170.0,1886.0,22255.0,647.200012,298.869995,195.970001,270.130005,102.370003,49.840000,...,44.669998,146.710007,118.720001,123.339996,49693.199219,7444.250000,63272.109375,29366.939453,2843.929932,17.870001
2963,2026-05-14,67970.0,1894.0,22155.0,640.000000,298.209991,192.660004,267.220001,102.070000,49.849998,...,44.900002,146.630005,118.669998,120.970001,50063.460938,7501.240234,62654.050781,29580.300781,2863.090088,17.260000
2964,2026-05-15,66610.0,1883.0,21695.0,639.700012,300.230011,180.330002,264.140015,92.239998,49.770000,...,43.869999,145.100006,116.529999,115.589996,49526.171875,7408.500000,61409.289062,29125.199219,2793.300049,18.430000


In [4]:
features = pd.read_csv("gold_etf_data/features.csv")
features

,Date,gold_etf_jp_1326_ret_1d,japan_reit_etf_ret_1d,gold_etf_jp_1540_ret_1d,japan_bank_etf_ret_1d,apple_ret_1d,agnico_eagle_ret_1d,amazon_ret_1d,anglogold_ret_1d,bank_of_america_ret_1d,...,year,month,day,dayofweek,business_day_in_month,business_days_in_month,remaining_business_days_in_month,month_progress,is_month_start_area,is_month_end_area
0,2015-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2015,1,1,3,1,22,21,0.045455,1,0
1,2015-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2015,1,2,4,2,22,20,0.090909,1,0
2,2015-01-05,NaN,NaN,NaN,NaN,-0.028172,0.016667,-0.020517,0.044419,-0.029050,...,2015,1,5,0,3,22,19,0.136364,1,0
3,2015-01-06,-0.002171,NaN,-0.006608,-0.026042,0.000094,0.057559,-0.022833,0.062159,-0.029920,...,2015,1,6,1,4,22,18,0.181818,0,0
4,2015-01-07,0.004351,NaN,0.006652,-0.010695,0.014022,-0.020324,0.010600,-0.018480,0.004745,...,2015,1,7,2,5,22,17,0.227273,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2961,2026-05-12,0.008471,-0.005759,0.006361,0.020220,0.007243,0.001067,-0.011785,-0.037013,0.004550,...,2026,5,12,1,8,12,4,0.666667,0,0
2962,2026-05-13,0.004568,-0.006846,0.004740,0.009988,0.013806,-0.005481,0.016214,-0.018787,-0.018511,...,2026,5,13,2,9,12,3,0.750000,0,1
2963,2026-05-14,-0.002934,0.004242,-0.004493,-0.011125,-0.002208,-0.016890,-0.010773,-0.002931,0.000201,...,2026,5,14,3,10,12,2,0.833333,0,1
2964,2026-05-15,-0.020009,-0.005808,-0.020763,-0.000469,0.006774,-0.063999,-0.011526,-0.096306,-0.001605,...,2026,5,15,4,11,12,1,0.916667,0,1
